In [2]:
# Configuración de entorno
ENTORNO = "colab" # Opciones: "local" o "colab"
ALMACENAMIENTO = "drive" # Opciones: "local" o "drive" (solo aplica si ENTORNO = "colab")

# Rutas base según la configuración
RUTA_DRIVE = '/content/drive/MyDrive/TFM'
RUTA_LOCAL = 'C:/Users/Usuario/Desktop/TFM_Mateu/gnn-material-science' # o ruta absoluta si se desea

import os
import sys

if ENTORNO == "colab":
    print("Ejecutando en entorno Google Colab...")
    dispositivo = "cuda"
    if ALMACENAMIENTO == "drive":
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            os.chdir(RUTA_DRIVE)
            print(f"Montado Drive y trabajando en {RUTA_DRIVE}")
        except ImportError:
            print("No se pudo montar drive. ¿Seguro que estás en Colab?")
    else:
        print("Usando almacenamiento local de la sesión de Colab.")
    
    # Instalamos dependencias si estamos en Colab
    print("Comprobando e instalando dependencias (puede tardar un poco)...")
    get_ipython().system('pip install -q -r requirements.txt')
    get_ipython().system('pip install -q cuequivariance cuequivariance-torch cuequivariance-ops-torch-cu12')
elif ENTORNO == "local":
    os.chdir(RUTA_LOCAL)
    dispositivo = "cpu" # Forzamos CPU para local según especificaciones
    print(f"Ejecutando en entorno local usando {dispositivo.upper()}. Directorio actual: {os.getcwd()}")
    if 'google.colab' in sys.modules:
        print("ADVERTENCIA: Parece que estás en Colab pero has seleccionado entorno 'local'.")
    
    print("Comprobando e instalando dependencias locales (puede tardar un poco)...")
    get_ipython().system('pip install -q -r requirements.txt')
else:
    print("Entorno no reconocido. Usa 'colab' o 'local'.")
    dispositivo = "cpu"


Mounted at /content/drive
Montado Drive y trabajando en /content/drive/MyDrive/TFM
Comprobando e instalando dependencias (puede tardar un poco)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 35.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.2/363.2 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 97.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA'}")

CUDA disponible: True
GPU: Tesla T4


In [4]:
# ============================================================
# PARAMETROS DE SIMULACION
# ============================================================
pasos = 5000

# Modo de progreso:
#   "log" -> muestra la tabla de energia/temperatura cada step (bueno para diagnostico)
#   "bar" -> muestra una barra de progreso visual (mas limpio para simulaciones largas)
modo_progreso = "bar"  # Opciones: "log" o "bar"

import sys
import subprocess

comando = [
    sys.executable, "-u", "MACE/npt.py",
    "--device", dispositivo,
    "--steps", str(pasos),
    "--progress", modo_progreso,
]
print(f"Ejecutando: {" ".join(comando)}\n")

# Usamos subprocess para forzar la impresion linea a linea en tiempo real
proceso = subprocess.Popen(comando, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for linea in proceso.stdout:
    print(linea, end="")

proceso.wait()
if proceso.returncode != 0:
    print(f"\nError: El proceso termino con codigo {proceso.returncode}")


Ejecutando: /usr/bin/python3 -u MACE/npt.py --device cuda --steps 5000 --progress bar

/usr/bin/python3: can't open file 'MACE/npt.py': [Errno 107] Transport endpoint is not connected

Error: El proceso termino con codigo 2
